# 01 — Indexing Pipeline

Everything that only needs to run **once** (or when documents change):

1. Load markdown docs and metadata from `md_docs/` and `md_ref/`
2. Chunk each document with header-aware splitting
3. Embed chunks with the local `bge-m3` model (dense vectors)
4. Encode with BM25 (sparse vectors)
5. Upsert into the Qdrant vector store

> Run this notebook once to populate the database.  
> After that, open `02_retrieval.ipynb` to run queries.

## 1 — Imports & Config

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from loguru import logger

from scientific_rag.ingestion.chunker import load_and_chunk_all, get_all_stems
from scientific_rag.vectorstore.embedders import LocalEmbedder
from scientific_rag.vectorstore.indexer import VectorStore

load_dotenv()

import sys
sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src")) if "__file__" in dir() else None

# ── Chunking ──────────────────────────────────────────────────────────────────
MAX_CHUNK_SIZE = 1200   # characters per chunk
CHUNK_OVERLAP  = 100    # overlap between consecutive chunks

# ── Qdrant ────────────────────────────────────────────────────────────────────
COLLECTION_NAME = "technical_docs"
QDRANT_URL      = "http://localhost:6333"
USE_HYBRID      = True   # dense + BM25 sparse vectors

print("Config ready.")
print(f"  Collection : {COLLECTION_NAME}")
print(f"  Hybrid     : {USE_HYBRID}")
print(f"  Chunk size : {MAX_CHUNK_SIZE} chars  |  overlap: {CHUNK_OVERLAP}")

## 2 — Inspect Available Documents

In [ ]:
stems = get_all_stems()

print(f"Found {len(stems)} documents in md_docs/:\n")
for s in stems:
    print(f"  • {s}")

## 3 — Chunk All Documents

In [ ]:
all_chunks = load_and_chunk_all(
    max_chunk_size=MAX_CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

print(f"\nTotal chunks ready to embed: {len(all_chunks)}")

sample = all_chunks[0]
print("\n── Sample chunk ─────────────────────────────────────────────────")
print(f"  stem    : {sample['metadata'].get('stem')}")
print(f"  title   : {sample['metadata'].get('title')}")
print(f"  section : {sample['metadata'].get('section_h2') or sample['metadata'].get('section_h1')}")
print(f"  chars   : {len(sample['text'])}")
print(f"  preview : {sample['text'][:200]}")

## 4 — Chunk Statistics

In [ ]:
from collections import Counter

per_doc = Counter(c['metadata']['stem'] for c in all_chunks)

print(f"{'Document stem':<55} {'chunks':>6}")
print("-" * 65)
for stem, count in sorted(per_doc.items()):
    print(f"  {stem:<53} {count:>6}")
print("-" * 65)
print(f"  {'TOTAL':<53} {sum(per_doc.values()):>6}")

lengths = [len(c['text']) for c in all_chunks]
print(f"\nChunk length — min: {min(lengths)}  max: {max(lengths)}  avg: {round(sum(lengths)/len(lengths))}")

## 5 — Load Embedder

In [ ]:
# Downloads the model on first run (~2 GB), then loads from cache.
embedder = LocalEmbedder()

print(f"Embedder model : BAAI/bge-m3")
print(f"Vector size    : {embedder.vector_size}")

## 6 — Connect / Create Qdrant Collection

> ⚠️ Set `FORCE_RECREATE = True` **only** if you want to wipe the collection and rebuild.  
> Leave it as `False` to connect to the existing collection without re-indexing.

In [ ]:
FORCE_RECREATE = True   # ← change to True to rebuild from scratch

store = VectorStore(
    embedder=embedder,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
    use_hybrid=USE_HYBRID,
    force_recreate=FORCE_RECREATE
)

info = store.collection_info()
print(f"\nCollection '{COLLECTION_NAME}' — {info.points_count} vectors currently stored")

## 7 — Index Chunks

Embeds every chunk (dense + sparse) and upserts into Qdrant.  
Skips automatically if the collection already has data and `FORCE_RECREATE = False`.

In [ ]:
import time

if info.points_count > 0 and not FORCE_RECREATE:
    print(f"Collection already has {info.points_count} vectors. Skipping indexing.")
    print("Set FORCE_RECREATE = True in cell 6 and re-run to rebuild.")
else:
    print(f"Indexing {len(all_chunks)} chunks...")
    t0 = time.time()
    store.index_chunks(all_chunks, batch_size=32)
    elapsed = round(time.time() - t0, 1)
    print(f"\nDone in {elapsed}s")

    info = store.collection_info()
    print(f"Vectors in collection: {info.points_count}")

## 8 — Sanity Check: Run a Test Query

In [ ]:
TEST_QUERY = "What methods are used to measure runway friction?"

results = store.search(TEST_QUERY, top_k=5)

print(f"Query: {TEST_QUERY}\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] score={round(r['score'], 4)}  stem={r['metadata'].get('stem', '')[:50]}")
    print(f"    section: {r['metadata'].get('section_h2') or r['metadata'].get('section_h1', '')}")
    print(f"    preview : {r['text']}")
    print()

## 9 — Collection Summary

In [ ]:
info = store.collection_info()

print("=" * 60)
print("COLLECTION SUMMARY")
print("=" * 60)
print(f"  Name          : {COLLECTION_NAME}")
print(f"  Vectors stored: {info.points_count}")
print(f"  Hybrid mode   : {USE_HYBRID}")
print(f"  Documents     : {len(stems)}")
print(f"  Chunks built  : {len(all_chunks)}")
print("=" * 60)
print("\n✓ Index is ready. Open 02_retrieval.ipynb to run queries.")